# Глава 9. Алгоритмическая оптимизация
В этой главе поговорим про сущетсвующие модификации трансформерных моделей, применяющиеся для ускорения обучения и инференса. Акцент именно на алгоритмические хаки, про ускорение с помощью железа и параллелизации будем говорить в следующих главах. Обзор на методы борьбы с квадратичным вниманием, поговорим про FlashAttention и затронем спекулятивный инференс.

## Введение
В 4 главе мы отмечали, что при всех плюсах у трансформерной разитектуры есть два важных недостатка

1) Самовнимание считает попарные взаимодействия всех токенов, то есть его стоимость по вычислениям и памяти растёт как $O(n^2)$ по длине последовательности $n$. Пока контекст короткий, это незаметно, но на больших контекстах (длинных документах, репозиториях кода и многошаговых агентских диалогах) квадратичная сложность становится существенным ограничением, проявляющимся на этапе обучения и на префилле

2) Во время инференса токены порождаются по одному, каждый требует отдельного прохода модели, и на каждом шаге из медленной памяти (HBM) приходится заново вычитывать веса модели и весь накопленный KV-кэш. Здесь упираемся в пропускную способность памяти: тензорные ядра простаивают, GPU большую часть времени ждёт данные. Размер KV-кэша при этом линейно растёт с длиной контекста и размером батча

## Эффективный Attention
В 2020 году было несколько попыток победить квадратичную сложность. Базовая идея у всех общая: матрица внимания n×n на практике почти везде «лишняя», потому что softmax концентрирует вес на небольшом числе ключей, а нужные связи часто либо локальны, либо низкоранговы. Различаются методы тем, как именно они сокращают эту матрицу

### Longformer
[[Beltagi et al, 2020]](https://arxiv.org/abs/2004.05150) пошли вероятно по самому простому пути, решили ограничить множество токенов, которые участвуют в подсчете внимания. Благо для этого в Трансформере уже есть готовый механизм - маска внимания. Если это сделать, то сохранив общее кол-во активных токенов, доступный контекст можно серьезно расширить. Модель назвали Longformer = Long Document Transformer, чтобы как раз подчеркнуть основную цель данной модификации - увеличение доступного контекста. 

Авторы расммотрели несколько масок внимания. Во-первых, это скользящее окно из соседних токенов (локальное внимание). Во-вторых, окно предложили разреживать (dilated window), то есть используем каждый второй или каждый третий токен и т.п. Да, есть риск пропустить какой-то важный токен, но с большой вероятностью его конткст подскажет. В третьих, некоторые токены получают безусловное внимание, так как считаются важными, к таким относятся токены из самого начала контекста. 

Сложность становится линейной по n.

Иллюстрация масок ниже. В режиме авторегресионной генерации (GPT) дополнительно добавляется нижнетреугольная маска:

<img src="img/longformer.png" width=500>

### BigBird
[[Zaheer, 2021]](https://arxiv.org/abs/2007.14062) решили, что важно расширить маски внимания случайными токенами.

<img src="img/bigbird.png" width=500>

Теоретическое обоснование взято из теории графов: графами «Small World» (графами высокой связности) называют графы, где путь между любой парой вершин короткий. Добавление случайных рёбер в граф довольно быстро превращают малосвязный граф в сильно связванный (как бы добавляется возможность "телепортации" из вершины в вершину). Этот прицип является основой многих социальных эффектов, в частности теории шести рукопожатий.

На уровне одного слоя внимания какой-то особенной связности нет: запрос стандартно агрегирует сигнал от доступных ему токенов k. Но если мы рассматриваем модель целиком, как суперкпозицию множества слоев внимания, то видим, как сигнал может распространяться от токена к токену в том числе через добавленные случайные связи. 

Авторы показали, что такая схема приближает выразительность полного внимания, при этом оставаясь линейной по времени вычисления.



<img src="img/routing.png" width=450>

### Reformer
[[Kitaev et al, 2020]](https://arxiv.org/abs/2001.04451) из Google обратили внимание, что сам расчет внимания в Трансформерной архитектуре избыточен и предложили свою архитектуру, которую назвали Reformer (Reversable Transformer). В ней они реализовали три модификации.

Во-первых, они заметили что на практике внимание распределяется неравномерно, обычно оно достается всего нескольким токенам из контекста, а значение остальных оказывается около нуля. Соотвественно нас в первую очередь интересуют токены дающие максимальное произведение, а остальными можно пренебречь. Максимум скалряного произведения достигается, когда q и k сонаправлены, иными словами, чем ближе векторные представления k и q, тем больше связь между ними. Если сегментировать вектора и распредлеить их в однородные "корзины", тогда перебор можно ограничить рамками одной корзины.

Для этого они прибегли к идеально подходящему под эту задачу алгоритму приближенного поиска соседей LSH (locality-sensitive hashing). Идея в том, что все ветктора описываются небольшим набором случайными проекцияй, которые можно очень быстро посчитать и не тратить время на полный расчет расстояния между векторами. Тогда с большой вероятностью близкие или идентичные представления попадут в одним проекций. Останется перебрать пары из одной корзины.

<img src="img/reformer.png" width=600>

Во-вторых, обучение сети методом обратного распролстранения ошибки требует хранения активаций всех слоев, поэтому стандартно на прямом проходе оптимизатор их запоминает в памяти и на обратном проходе достает. Но если хочется сэкономить на памяти, на обратном проходе активации можно не запомнианть, а вычилсять повторно - это занимает больше времени, но меньше расходуется память. В случае с классическим Трансофрмером это невозможно, так как . Поэтому авторы предложили переконфигурировать остаточные связи - . Теперь тут нет рекурсии и взоды можно восстановить. Назали это обратимыми слоями (reversible residuals).

Третья новация связана с вычислением полносвязного слоя (FFN). Это вычисление использует самый большой тензор, что дает пиковое использование памяти. Но поскольку применяется независимо ко всем токенам входной последовательности, вычисление можно разбить на последовательные блоки. Общее кол-во не меняется, но пиковое испольование памяти уменьшается.

Статья дала старт целому направлению "Efficient Transformers", но мейнстримом модель так и не стала, уступив пальму первенства методам типа FlashAttention (о нем ниже).

### Routing Transformer
[[Roy et al, 2020]](https://arxiv.org/abs/2003.05997) вышли с похожей на Reformer идеей - сделать маску разреженности динамической, зависящей от входа, но вместо LSH они использовали кластеризацию. Модель назвали __Routing Transformer__, показывая, что. В качестве метода кластеризации использовали k-means (онлайн его вариант: примеры показываются один за другим, кластеризация уточняется). Каждый запрос $q$ считает внимание только с ключами $k$ своего же кластера и таким образом сокращается общее кол-во вычислений.

### Linformer
Для экономии вычислений [[Wang et al, 2021]](https://arxiv.org/abs/2006.04768) идут чуть по другому пути, они не разделяют последовательность токенов на сегменты, а плотно упаковывают всю эту последовательность представлений в небольшой фиксированной размерности вектор псевдотокенов. Делается это для представлений K и V через два добавленных перед вниманием слоя проекций, после чего вычисляется стандартный механизм внимания. В батчевом режиме он уже не квадратный, а прямоугольный $QK_{dense}^T$, то есть каждый оригинальный токен $q$ агрегирует сигнал от сжатого набора псевдотокенов $k$.

Модель назвали __Linformer__ акцентируя внимание на том, что из-за фикисрованной размерности последовательности псевдотокенов, сложность вычисления становится линейной: ведь независимо от размера входного контекста, нам всегда достаточно сравниться с k псевдотокенами.

<img src="img/linformer1.png" width=500>

### Performer
Команда [[Choromankspiy et al, 2022]](https://arxiv.org/abs/2009.14794) из Google также пошли по пути приближения, но более радикально, они предложили заменить саму формулу расчета внимания на ее приближенный вариант, основываясь на паре математически приемов, главным из которых было разложении ядра (kernel trick). Модель назвали __Performer__, а алгоритм приближенного внимания FAVOR+ (Fast Attention Via positive Orthogonal Random features). В результате упрощения формулы появилвась ассоциативность и умножать на V раньше, чем умножатьна Q. Тем самым вообще не материализовать квадратичную матрицу $QK^T$.

Напомним, что обычное внимание вычисляется как $\mathrm{Att}(Q,K,V)=D^{-1}AV$, где $A_{ij}=\exp(q_i^\top k_j/\sqrt d)$ и $D=\mathrm{diag}(A\mathbf 1_L)$. 

Построение матрицы $A$ размера $L\times L$, имеет квадратичную по длине контекста $L$ сложность. Авторы $A_{ij}=\mathcal K(q_i,k_j)$ предлагают представление $\mathcal K(q,k)=\mathbb E_{\omega}[\varphi_\omega(q)^\top\varphi_\omega(k)]$ через отображение в признаковое пространство. Если построить явное приближение $\varphi:\mathbb R^d\to\mathbb R^m$, то $A\approx\varphi(Q)\varphi(K)^\top$, и матрица факторизуется на два «узких» множителя.

Основная сложность в том, что не очень понятно, как выбирать конкретное отображение $\varphi$ в пространство фичей. По классике для разложения RBF ядра (экспоненты) используют тригонометрические случайные признаки:

$$\varphi_{\mathrm{trig}}(x) = \frac{1}{\sqrt{m}}\Big[\cos(\omega_1^\top x),\dots,\cos(\omega_m^\top x),\; \sin(\omega_1^\top x),\dots,\sin(\omega_m^\top x)\Big], \qquad \omega_i \stackrel{\text{iid}}{\sim} \mathcal{N}(0,I_d)$$

Но проблема с ними в том, что они дают знакопеременные координаты, из-за чего сложно приближать малые значения ядра — а именно такие значения доминируют в реальных матрицах внимания. Поэтому авторы предложили использовать другое представление со строго положительными признаками:

$$\varphi(x)=\frac{\exp(-\|x\|^2/2)}{\sqrt m}\Big[\exp(\omega_1^\top x),\dots,\exp(\omega_m^\top x)\Big],\qquad \omega_i\sim\mathcal N(0,I_d),$$

для которого $\mathbb E[\varphi(q)^\top\varphi(k)]=\exp(q^\top k)$ точно, а дисперсия стремится к нулю там же, где стремится к нулю само ядро. 

Дополнительно векторы $\omega_i$ генерируются попарно ортогональными, что строго уменьшает дисперсию оценки.

После замены $QK^T$ на приближенное представление становится возможной перестановка скобок в произведении, где вместо $(\varphi(Q)\varphi(K)^\top)V$ мы можем посчитать $\varphi(Q)\,(\varphi(K)^\top V)$. Знаменатель находится аналогично: $\hat D=\mathrm{diag}\big(\varphi(Q)(\varphi(K)^\top\mathbf 1_L)\big)$. То есть не материализовать проблемную квадратную матрицу, в просто считать проивзедение двух векторов. В авторегрессионном режиме внимание вообще обновляется за константное время $O(1)$, а длина последовательности нигде не зашита в веса, что делает подход гибким.

$$\mathrm{out}_i=\hat d_i^{-1}\,\varphi(q_i)^\top S_i,\qquad S_i=S_{i-1}+\varphi(k_i)v_i^\top\in\mathbb R^{m\times d},$$

На практике замена не очень окупилась: Performer уступил полному вниманию, поскольку приемлемая дисперсия требует довольно большой размерности $m$. Тем не менее важный пример того, что линейное внимание может быть выведено из аппроксимационных соображений. Похожий прием испольщовался в более поздней модели MAMBA (подробнее в следующей главе).

## Экстраполяция конекста
### Интерпоялция позиций
### NTK-scaling
### YARN

## Экономия KV-кэша
Второе слабое место Траснформерной архитектуры - это инференс, потокенная генерация ответа. Мы кэшируем ключи и значения всех прошлых токенов. Размер кэша пропорционален числу слоёв, числу голов, размерности головы, длине контекста и батчу — и именно он определяет, сколько памяти и пропускной способности съест декодирование.

### MQA
[[Shazeer, 2019]](https://arxiv.org/abs/1911.02150) предложили сыграть на избыточности внимания и оставить только по одному экзмепляру проекций для K и V векторов. Разнообращие обьеспечивается только для запросов Q, поскольку считает, наиболее важный компонент сигнала, для Q оригинальное количество представлений. Таким образом общее количество вычислений не меняется, но использование памяти под KV кэш заметно сокращается, а значит и генерация заметно ускоряется. Подход назвали __Multi-Query Attention__. За оптимизацию приходится платить некоторой просадкой качества и меньшей стабильностью обучения, поскольку ограничивается разнообразие представлений K и V.

<img src="img/mqa.webp" width=500>

### GQA
[[Ainslie et al, 2023]](https://arxiv.org/abs/2305.13245) предложили __Grouped-Query Attention__ компромисс между полным вниманием (MHA) и MQA. Можество запросов делятся на G групп, и каждая группа разделяет одну голову K/V. существующую MHA-модель можно дёшево «дообучить» (uptrain) в GQA. В том числе по этой причине GQA стал стандартом в моделях LLaMA-2/3, Mistral и других.

<img src="img/gqa.png" width=500>

### MLA
[[Liu et al, 2024]](https://arxiv.org/abs/2405.04434) __Multi-head Latent Attention__ (DeepSeek-V2) меняет сам вопрос. Вместо «как поделить меньшее число голов K/V» спрашивается «зачем вообще хранить полноразмерные K и V». K и V совместно сжимаются низкоранговой проекцией в маленький латентный вектор, и в кэше лежит только он; при вычислении внимания пер-головые K и V восстанавливаются обратной проекцией на лету. В DeepSeek-V2 это дало сокращение KV-кэша примерно на 93% относительно MHA, причём, по их измерениям, не ценой качества, а с небольшим выигрышем. 

Платой стала сложность: низкоранговое сжатие плохо дружит с RoPE, поэтому введён «расщеплённый» (decoupled) RoPE — отдельная небольшая часть размерностей несёт позиционную информацию; и приём «поглощения весов» (weight absorption), сворачивающий проекции, чтобы восстановление не стоило лишних вычислений. MLA архитектурно более инвазивен, чем GQA, но и потенциально мощнее: он торгует дополнительными вычислениями (распаковка) за резкое снижение памяти и трафика

<img src="img/mla.png" width=500>

### NSA
Команда [(Yuan et al, 2025)](https://arxiv.org/abs/2502.11089) из DeepSeek вернулись к старой идее разреженности со своим вариантом внимания __NSA (Native Sparse Attention)__. Они сформулировали пайплайн, состоящий из трех парадлельных экстракторов. Во-первых, это сжатый поиск: блоки токенов суммаризируются в компактные представлений. Во-вторых, точный поиск: выбираются самые релевантные блоки токенов, к которым затем применяется полное внимание. В третьих используется скользящее окно (свежий локальный контекст). Выходы всех трех компонентов смешиваются обучаемым гейтом. 

Два принципиальных отличия от методов 2020 года. Во-первых, NSA обучаема нативно — разреженность присутствует с самого предобучения, а не плявляется в первый раз на инференсе. Во-вторых, она аппаратно-согласована: шаблон спроектирован под GPU (сбалансированная арифметическая интенсивность, блочные ядра, выровненные под группировку GQA), поэтому теоретическая экономия операций превращается в реальное ускорение по времени. На последовательностях в 64k NSA заметно быстрее полного внимания на декодировании, прямом и обратном проходах, при этом не уступая ему в качестве.

<img src="img/nsa.png" width=500>

## FlashAttention
(Dao et al, 2022) задались вопросом, можно ли перекомпоновать алгоритм расчета, оставив его точным, но ускорить вычисление. Оказалось, что не только можно, но это еще и очень эффективно. Так родился алгоритм __FlashAttention__ модификация обычного самовнимания, ускоряющая расчет. Ключевое наблюдение — стандартное внимание упирается не в вычисления, а в память: оно записывает в медленную HBM огромную промежуточную матрицу n×n и читает её обратно. Значит, надо минимизировать обращения к HBM, а не число операций.

### FlashAttention-1
[[Dao et al, 2022]](https://arxiv.org/abs/2205.14135)<Br>
FlashAttention компонует вычисление внимания таким образом, что становится более оптимальным по I/O. На GPU есть быстрая SRAM память и медленная HBM, хочется больше вычислений делать на SRAM, 

Attention матрица нарезается на куски (процесс называется тайлинг): блоки Q, K, V подгружаются из HBM в быструю память SRAM и обрабатываются по частям. 
Softmax считается «онлайн» по частям (с бегущими максимумом и суммой), поэтому полная матрица внимания нигде не материализуется целиком. 
На обратном проходе используется пересчёт: вместо хранения большой матрицы её восстанавливают из компактной статистики. 

В итоге память линейна по n, число обращений к HBM резко падает, а итоговое ускорение по времени — в 2–4 раза, почти бесплатно и с сохранением точности.

<img src="img/flash1.png" width=600>

### FlashAttention-2
[[Dao et al, 2023]](https://arxiv.org/abs/2307.08691)<br>
__FlashAttention-2__, вторая версия (2023), не меняет алгоритм, но переписывает распараллеливание. Сокращается доля «не-matmul» операций (тензорные ядра GPU считают матричное умножение во много раз быстрее, чем специальный блок, отвечающий за экспоненту в softmax), вычисление параллелится вдоль длины последовательности, а работа лучше распределяется между варпами и блоками, уменьшая трафик через разделяемую память. Это даёт ещё около двукратного ускорения и доводит утилизацию до примерно 50–70% на A100. На H100, однако, версия достигала лишь ~35%, потому что не использовала особенности нового железа

### FlashAttention-3
В 2024 году [[Shah et al, 2024]](https://arxiv.org/abs/2407.08608) выпустили __FlashAttention-3__, третью версия алгоритма. В этот раз целью была максимально возможная синхронизацию с железом, а конкренто с архитектурой Nvidia Hopper (H100). Авторы при этом не отходят от своего главного принципа - внимание по-прежнему остается точным.

Три приёма: использование асинхронности (warp-specialization — одни варпы через TMA асинхронно подгружают данные, другие в это время считают на тензорных ядрах WGMMA, перекрывая память и вычисления); чередование (ping-pong) блочного matmul и softmax, чтобы медленная экспонента считалась одновременно с матричным умножением; и низкая точность FP8 с блочным квантованием и «incoherent processing», которые удерживают точность (примерно в 2,6 раза меньше ошибка, чем у наивного FP8). 

В результате простой на видеокартах H100 сокращается с 65% до 25%, а скорость генерации становится в 1.5–2 раза выше второй версии.

Алгоритм FlashAttention совершил своего рода революцию в вычислениях языковых моделей на видеокартах и фактически обесценил целое направление приближённых методов,  и показав, что того же эффекта можно добиться с сохранением точности и сделав их бесполезными.

## Спекулятивное декодирование
[(Leviaithan et al, 2023)](https://arxiv.org/pdf/2211.17192) описали идею спекулятивного декодирования (speculative decoding), основой которой является наблюдение, что генерация обычно неравномерна по сложности. Какие-то куски ответа сгенерировать просто (```2 x 2 = ```), для каких-то нужно серьезное рассуждение. Почему бы не переключаться между разными моделями на ходу? Допустим есть большая языковая модель (целевая, target), и есть компактная языковая модель (черновая, draft). Дадим малой модели возможность быстро генерировать продолжение на несколько шагов, а затем большая модель проверит качество ее продолжения. Если оно удовлетворительное, то идем дальше, если неудовлетворительное, целевая модель перегенерирует то же самое продолжение, но уже самостоятельно.

Шаг верификации продолжения дешевый, поскольку происходит за одну итерацию Траснформера может проверяться сразу много нагенерированных черновых токенов. В роли модели-черновика часто выбирают более базовую версия той же целевой модели. Приемка вероятностная: целевая модель сравнивает две вероятности сегенерированного продолжения, свою $P(t)$ и дочерней модели $Q(t)$. Если дочерняя модель переоценила вероятность токена $(Q > P)$, продолжение принимается с вероятностью $P/Q$. Если недооценила, заменяем на вариант целевой модели.

Таким образом, выигрыш есть, если черновик дёшев, а доля принятых токенов высока.

### Blockwise Parallel Decoding
В 2018 году еще до появления самого термина "спеулятивное декодирование" описали модель Blockwise Parallel Decoding. Идея простая - к выходу последнего скрытого состояния модели добавляется несколько лёгких «голов» (heads), каждая из которых независимо предсказывает токен на позиции +1, +2, +3 и так далее.

### Medusa
В области генеративных языковых моделей существует целый набор методов, озаглавленный неавторегрессионная генерация (NAT), основная идея которого - генерация продолжения не из одного, а сразу из сразу нескольких токенов. 
Первая модель была вообще без верификации . Позже появилась модель Blockwise Parallel Decoding. 

[[Cai et al, 2024]](https://arxiv.org/abs/2401.10774) взяли старую модель BPD и предложили пару модификаций процесса верификации. Главная доработка - вместо генерации одного продолжения генерируется сразу множество и организуется в виде дерева. Так появилась модель __Medusa__.  В первой версии модели Medusa-1 достаточно обучить только головы, остальные веса замораживается, что обсепечивает высокую скорость работы. Во второй версии модель обучается целиком.

Как генерируется дерево продолжений? Каждая голова генерирует top-k токенов и по этим наборам строится декартово произведение всех возможных комбинаций, на основании которой строится дерево. Далее все комбинации укладываются в одну плоскую структуру - это сделано, чтобы можно было посчитать вероятности каждого токена за одну итерацию. А чтобы гарантировать, что токен видит только свой префикс используется маска внимания. В MEDUSA такую маску внимания называют Tree Attention. И затем остается верифицировать все сгенерированные продолжения, выбрав наиболее вероятное.

<img src="img/medusa1.png" width=300>

В работе описано три механизма верификации. В рамках "жадной" стратегии, мы просто на каждом шаге выбираем токен с наибольшей вероятностью по целевой модели. В рамках стратегии "Typical Acceptance" мы считаем вероятность каждого токена по основной модели и сравниваем ее с порогом уверенности, скорректированным на энтропию его распределния (при малой энтропии порог уменьшается). В рамках стратегии `nucleus` для каждой позиции в дереве алгоритм берёт распределение вероятностей всех токенов, полученное от основной модели, сортирует их по убыванию и начинает добавлять токены в так называемое «ядро», пока суммарная вероятность накопленных токенов не достигнет порога `top_p` (обычно 0.8). Токен-кандидат от головы MEDUSA принимается, если он попал в это ядро, и отклоняется в противном случае.

Явный минус подхода MEDUSA в том, что каждая голова генерирует токен независимо от других (находятся вне локального контекста, особенно последние токены), поэтому точность генерации явно падает.

### Hydra
[(Ankner, 2024)](https://arxiv.org/abs/2402.05109) решили исправить эту независимость и сделали генерирующие головы (heads) последовательно зависимыми: каждая получает на вход токены, предложенные предыдущими. По сути черновик превращается из набора независимых предсказанных токенов в нормальную последовательную модель, что заметно повышает среднюю длину принятого фрагмента.

<img src="img/hydra.png" width=300>

### EAGLE
[[Li et al, 2024]](https://arxiv.org/abs/2401.15077) ключевая мысль первой версии модели __EAGLE-1__ (Extrapolation Algorithm for Greater Language-model Efficiency): генерировать следующий токен не по токеном, но и скртые преставение  предпоследнего скрытого состояния. Идея в том, что непрерывное скрытое представление гораздо лучше в качестве сигнала, чем дискретные токены. затем из предсказанного признака получают токен через готовую LM-голову целевой модели. Скрытое состояние передается вместе со сгенерированным токеном.

Черновик делает фиксированное число авторегрессионных шагов, обычно 4-8. Затем верификатор (полнаая модель) оценивает и оставляет наиболее длинную принятую последовательность. Черновая модель представляет сильно усеченную версию основной модели: слой эмбединов и выходной HEAD слой берутся из оригинальной модели, а между ними добавляется один траснформеный слой. Модель в авторегрессионном режиме генерирует продолжение и . Для верификации заимствуется идея из MEDUSA, генерируется сразу дерево возможных продолжений, которые оцениваются механикой TreeAttetnion.

<img src="img/eagle1_1.png" width=500>

[(Li et al, 2024)](https://arxiv.org/abs/2406.16858) во второй версии модели __EAGLE-2__ корректировку решили делать не после генерации чернового дерева, а в процессе его построения с помощью beam search. Черновая модель сама оценивает уверенность в ответе.

[[Li et al, 2025]](https://arxiv.org/abs/2503.01840) в версии __EAGLE-3__ решили отказаться от предсказания скрытого состояния, а предсказывать сразу токен. Вместо одних только верхних признаков используется конкатенация признаков с разных слоев. Кроме того, многошаговый процесс черновика симулируется уже на обучении, устраняя рассинхрон между обучением и инференсом. Это даёт порядка 3–6,5× ускорения относительно обычной генерации и на 20–40% выше EAGLE-2.


С чем они себя сравнивают:
- Transformer-XL (2019)
- Adaptive Span (2019)
- Compressive (2020)
- Reformer (2020)
- Sparse (2019)
- Routing (2020)
- BP-Transformer (2019)
- Blockwise (2019)